# **SQL Database Python Connection**

In [1]:
# Setting imports for reusable python script of music genre classification project

import sqlite3
import os
import pathlib
import pandas as pd
from sklearn.model_selection import train_test_split

print(pathlib.Path.cwd())
print("Imports successful!")

c:\Users\winni\music-genre-class\music-genre-classification
Imports successful!


In [3]:
# Python reproducible script to populate database table

GTZAN_ROOT = str(pathlib.Path.cwd() / "Data_Music" / "genres_original")
GENRES = sorted(os.listdir(GTZAN_ROOT))
LABEL_MAP = {genre: i for i, genre in enumerate(GENRES)}
DB_PATH = str(pathlib.Path.cwd() / "Data_Music" / "gtzan_metadata.db")

rows = []
for genre in GENRES:
    folder = os.path.join(GTZAN_ROOT, genre)
    for fname in os.listdir(folder):
        if fname.endswith(".wav"):
            rows.append({
                "file_path": os.path.join("genres", genre, fname),
                "label": genre,
                "label_id": LABEL_MAP[genre],
            })

df = pd.DataFrame(rows)

# 70 / 15 / 15 split, stratified by label
train, temp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
val, test   = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=42)

train["split"] = "train"
val["split"]   = "val"
test["split"]  = "test"
df_final = pd.concat([train, val, test]).reset_index(drop=True)
df_final["is_valid"] = 1    # default all files to valid

con = sqlite3.connect(DB_PATH)
df_final.to_sql("audio_metadata", con, if_exists="replace", index=False)
con.close()
print(f"Inserted {len(df_final)} rows into {DB_PATH}")

Inserted 1000 rows into c:\Users\winni\music-genre-class\music-genre-classification\Data_Music\gtzan_metadata.db


In [5]:
# Testing query from notebook

con = sqlite3.connect(DB_PATH)

train_df = pd.read_sql("SELECT * FROM audio_metadata WHERE split='train' AND is_valid=1", con)
val_df   = pd.read_sql("SELECT * FROM audio_metadata WHERE split='val'   AND is_valid=1", con)
test_df  = pd.read_sql("SELECT * FROM audio_metadata WHERE split='test'  AND is_valid=1", con)

con.close()
print(train_df.shape, val_df.shape, test_df.shape)

(700, 5) (150, 5) (150, 5)
